# dismantle — FINAL PUSH (all heavy compute)

Single-stream decode is bandwidth-bound (~90 tps ceiling on M3 Pro for q3b; llama.cpp ~50). The only ways past that are the **multiplier** (spec tokens per weight-read) and the **denominator** (fewer bytes/token via sparsity / low-bit). This notebook trains every cloud piece of both, measured by **tps-predictive** metrics.

- **Track B (default ON):** maximize the spec head (accepted-prefix 1.6 → target 2.5+). This is the proven core heavy-compute.
- **Tracks C/D (default OFF, audit before enabling):** sparsity predictor + Q2 draft — experimental, depend on local capture.

**Recommended path:**
1. `Runtime → Run all` once with `SMOKE_MODE=True` in cell B0 → validates end-to-end in ~2 min.
2. Set `SMOKE_MODE=False` in cell B0, re-run cell B1 → full sweep (2-4 GPU-hours). Already-evaluated configs from prior runs are skipped automatically.
3. Best head per model + leaderboard land in Drive.

## A. Setup — Drive (output), repo, deps, inputs

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
OUT_ROOT = '/content/drive/MyDrive/dismantle_final_push'
import os; os.makedirs(OUT_ROOT, exist_ok=True)
FROZEN_CACHE = os.path.join(OUT_ROOT, 'frozen_cache')  # cross-session frozen cache
os.makedirs(FROZEN_CACHE, exist_ok=True)
REPO_URL='https://github.com/joshuahickscorp/dismantle.git'; BRANCH='codex/maximal-spec-colab'
if not os.path.isdir('/content/dismantle'):
    !git clone --depth 1 --branch $BRANCH $REPO_URL /content/dismantle
else:
    !cd /content/dismantle && git fetch --depth 1 origin $BRANCH && git checkout $BRANCH && git reset --hard origin/$BRANCH
!pip -q install pyarrow safetensors gguf packaging huggingface_hub
import sys; sys.path.insert(0,'/content/dismantle/colab')
# GPU detection — H100/A100 get a bigger batch for better utilization.
import subprocess as _sp
try:
    _gpu = _sp.run(['nvidia-smi','--query-gpu=name','--format=csv,noheader'],
                   capture_output=True, text=True, timeout=5).stdout.strip()
except Exception:
    _gpu = ''
GPU_NAME = _gpu or 'unknown'
BATCH_SIZE = 64 if any(x in GPU_NAME for x in ('H100','A100')) else 32
MAX_ROWS = 8000 if any(x in GPU_NAME for x in ('H100','A100')) else 6000
print(f'GPU={GPU_NAME!r}  BATCH_SIZE={BATCH_SIZE}  MAX_ROWS={MAX_ROWS}')
print('ready; output ->', OUT_ROOT)


### A2. Fetch corpora (release) + rebuild frozen (HF GGUF, verified, **Drive-cached**)
First session: downloads ~250MB corpora + ~12GB GGUFs, rebuilds frozen (~10 min). Subsequent sessions: restores frozen from Drive cache (~30s).

In [ ]:
REPO_SLUG='joshuahickscorp/dismantle'; RELEASE_TAG='headbank-corpus-v1'
MODELS={
    "q05b": {
        "capture_layer": 23,
        "label": "Qwen2.5-0.5B-Instruct",
        "hf_repo": "Qwen/Qwen2.5-0.5B-Instruct-GGUF",
        "hf_file": "qwen2.5-0.5b-instruct-q4_k_m.gguf"
    },
    "q1p5b": {
        "capture_layer": 27,
        "label": "Qwen2.5-1.5B-Instruct",
        "hf_repo": "Qwen/Qwen2.5-1.5B-Instruct-GGUF",
        "hf_file": "qwen2.5-1.5b-instruct-q4_k_m.gguf"
    },
    "q3b": {
        "capture_layer": 35,
        "label": "Qwen2.5-3B-Instruct",
        "hf_repo": "Qwen/Qwen2.5-3B-Instruct-GGUF",
        "hf_file": "qwen2.5-3b-instruct-q4_k_m.gguf"
    },
    "q7b": {
        "capture_layer": 27,
        "label": "Qwen2.5-7B-Instruct",
        "hf_repo": "bartowski/Qwen2.5-7B-Instruct-GGUF",
        "hf_file": "Qwen2.5-7B-Instruct-Q4_K_M.gguf"
    }
}
import os, json, tarfile, urllib.request, urllib.error, subprocess, shutil, time, numpy as np
from huggingface_hub import hf_hub_download
DATA_ROOT='/content/headbank_inputs'
BASE=f'https://github.com/{REPO_SLUG}/releases/download/{RELEASE_TAG}'
BUILDER='/content/dismantle/tools/orchestrator/build_frozen_gguf.py'
os.makedirs(DATA_ROOT, exist_ok=True)

def _retry_urlretrieve(url, dst, tries=3):
    last=None
    for i in range(tries):
        try: urllib.request.urlretrieve(url, dst); return
        except (urllib.error.URLError, OSError, TimeoutError) as e:
            last=e; time.sleep(2*(i+1))
    raise last

def _safe_extract(tar, path):
    try: tar.extractall(path, filter='data')  # py3.12+ safe
    except TypeError: tar.extractall(path)    # py<=3.11 fallback

fp_path=os.path.join(DATA_ROOT,'frozen_fingerprints.json')
_retry_urlretrieve(f'{BASE}/frozen_fingerprints.json', fp_path)
FP=json.load(open(fp_path))
READY=[]
for slug,cfg in MODELS.items():
    dst=os.path.join(DATA_ROOT,slug); shards=os.path.join(dst,'corpus_shards')
    os.makedirs(shards, exist_ok=True); frozen=os.path.join(dst,'frozen_gguf.npz')
    cached=os.path.join(FROZEN_CACHE, slug, 'frozen_gguf.npz')
    if not any(f.endswith('.parquet') for f in os.listdir(shards)):
        tarp=os.path.join(dst,'corpus.tar')
        print(f'{slug}: downloading corpus...', flush=True)
        _retry_urlretrieve(f'{BASE}/{slug}_corpus.tar', tarp)
        with tarfile.open(tarp) as t: _safe_extract(t, shards)
        os.remove(tarp)
    # Restore frozen from Drive cache if present (skips ~5GB HF download).
    if not os.path.isfile(frozen) and os.path.isfile(cached):
        print(f'{slug}: restoring frozen from Drive cache...', flush=True)
        shutil.copy2(cached, frozen)
    if not os.path.isfile(frozen):
        print(f'{slug}: downloading {cfg["hf_file"]} + rebuilding frozen...', flush=True)
        gguf=hf_hub_download(repo_id=cfg['hf_repo'], filename=cfg['hf_file'])
        r=subprocess.run(['python',BUILDER,'--gguf',gguf,'--out',frozen],capture_output=True,text=True)
        if r.returncode!=0:
            print(f'{slug}: build_frozen failed:', r.stderr[-300:]); continue
    # Frozen may be corrupted from a prior partial run; delete + skip on bad load.
    try:
        z=np.load(frozen)
        on=z['output_norm'].astype(np.float32)
    except Exception as e:
        print(f'{slug}: frozen unreadable ({type(e).__name__}: {e}); deleting'); os.remove(frozen); continue
    ref=np.array(FP[slug]['output_norm'],dtype=np.float32)
    if on.shape!=ref.shape:
        print(f'{slug}: SKIP — output_norm shape {on.shape} != fingerprint {ref.shape}'); continue
    diff=float(np.abs(on-ref).max())
    ok=diff<1e-3
    n=len([f for f in os.listdir(shards) if f.endswith('.parquet')])
    print(f'{slug}: shards={n} frozen_verified={ok} (max|on-ref|={diff:.2e})')
    if ok and n>0:
        READY.append(slug)
        # Cache verified frozen to Drive so the next session skips the HF download.
        os.makedirs(os.path.dirname(cached), exist_ok=True)
        if not os.path.isfile(cached):
            print(f'{slug}: caching frozen to Drive ({os.path.getsize(frozen)/1e6:.0f} MB)...', flush=True)
            shutil.copy2(frozen, cached)
    elif not ok:
        print(f'   !! {slug} SKIPPED — fingerprint mismatch. The local build of '
              f'{cfg["hf_file"]} differs from the one in the release.\n'
              f'   To train {slug} anyway: upload your matching {slug}_frozen.npz '
              f'to {FROZEN_CACHE}/{slug}/frozen_gguf.npz and re-run this cell.')
print('READY:',READY)


## B. HEAD MAXIMIZATION SWEEP  *(the core heavy compute)*
For each model, train a grid of head configs (chained-hidden rollout) and keep the one with the highest **accepted-prefix** (the runtime-predictive spec multiplier). Ordered cheap→expensive so a time-boxed run still produces a winner.

**Cell B0** sets toggles. Set `SMOKE_MODE=True` for a ~2 min end-to-end validation, then flip to `False` for the full sweep.
**Cell B1** runs it. Subprocess stdout STREAMS — you see progress as training runs. Configs with an existing `tau.json` are SKIPPED on resume (so a Colab disconnect mid-sweep doesn't re-train completed configs).

In [ ]:
# ── B0. Toggles ───────────────────────────────────────────────────
SMOKE_MODE = False  # True: ~2-min validation; False: full ~3 GPU-hour sweep
FULL_SWEEP = [[1, 4.0, "1,2,3,4", 12], [2, 4.0, "1,2,3,4", 12], [1, 4.0, "1,2,3,4,5,6", 14], [2, 4.0, "1,2,3,4,5,6", 14]]
SMOKE_SWEEP = [[1, 4.0, "1,2", 2]]
SMOKE_MODEL = 'q05b'
SMOKE_MAX_WINDOWS = 256
SMOKE_MAX_ROWS = 512
if SMOKE_MODE:
    SWEEP = SMOKE_SWEEP
    TARGETS = [SMOKE_MODEL] if SMOKE_MODEL in READY else READY[:1]
    EVAL_MAX_WINDOWS = SMOKE_MAX_WINDOWS
    TRAIN_MAX_ROWS = SMOKE_MAX_ROWS
else:
    SWEEP = FULL_SWEEP
    TARGETS = list(READY)
    EVAL_MAX_WINDOWS = 4000
    TRAIN_MAX_ROWS = MAX_ROWS
print(f'SMOKE_MODE={SMOKE_MODE}  TARGETS={TARGETS}  SWEEP_LEN={len(SWEEP)}  '
      f'BATCH_SIZE={BATCH_SIZE}  TRAIN_MAX_ROWS={TRAIN_MAX_ROWS}  '
      f'EVAL_MAX_WINDOWS={EVAL_MAX_WINDOWS}')


In [ ]:
# ── B1. Sweep ─────────────────────────────────────────────────────
import subprocess, os, json, sys, time
TRAINER='/content/dismantle/colab/eagle5_train_pytorch.py'
EVAL='/content/dismantle/colab/eagle5_tau_eval_pytorch.py'

def _run_streaming(cmd, label):
    """Run subprocess, streaming combined stdout+stderr line-by-line.
    Returns exit code. Visibility during long runs prevents 'silent'
    Colab cells; if the kernel disconnects, the lines already printed
    survive in the notebook output."""
    print(f'>>> {label}', flush=True)
    t0 = time.time()
    p = subprocess.Popen(cmd, stdout=subprocess.PIPE,
                         stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in iter(p.stdout.readline, ''):
        sys.stdout.write(f'  | {line}'); sys.stdout.flush()
    p.wait()
    print(f'<<< {label} rc={p.returncode} wall={time.time()-t0:.1f}s', flush=True)
    return p.returncode

BEST={}; LEADER=[]
for slug in TARGETS:
    cfg=MODELS[slug]; shards=os.path.join(DATA_ROOT,slug,'corpus_shards')
    frozen=os.path.join(DATA_ROOT,slug,'frozen_gguf.npz')
    best_prefix=-1.0; best_dir=None
    for cfg_idx,(nb,ffm,rt,ep) in enumerate(SWEEP, start=1):
        tag=f'b{nb}_ff{ffm}_rt{rt.replace(",","-")}_e{ep}'
        ckpt=os.path.join(OUT_ROOT,'sweep',slug,tag); os.makedirs(ckpt,exist_ok=True)
        tj=os.path.join(ckpt,'tau.json')
        # Skip resumed configs that already have a complete eval.
        if os.path.isfile(tj):
            try:
                d=json.load(open(tj)); prefix=float(d.get('tau',-1.0))
                print(f'[skip] {slug}/{tag} (resumed) accepted_prefix={prefix:.3f}', flush=True)
                LEADER.append({'slug':slug,'config':tag,'accepted_prefix':prefix,
                               'depth1':d.get('depth1_accept_rate'),'resumed':True})
                if prefix>best_prefix: best_prefix=prefix; best_dir=ckpt
                continue
            except Exception as e:
                print(f'[skip-warn] {slug}/{tag} tau.json unreadable ({e}); will retrain', flush=True)
        # Train.
        tr=['python','-u',TRAINER,'--corpus-dir',shards,'--frozen',frozen,'--ckpt-dir',ckpt,
            '--device','cuda','--target-mode','corpus','--capture-layer',str(cfg['capture_layer']),
            '--epochs',str(ep),'--batch-size',str(BATCH_SIZE),'--seq-len','16','--lr','1e-3',
            '--num-blocks',str(nb),'--head-ff-mult',str(ffm),
            '--rollout-loss-weight','1.0','--rollout-depth','8',
            '--rollout-depth-targets',rt,'--rollout-draft-prob','0.75',
            '--rollout-chain-hidden','--save-safetensors',
            '--max-rows',str(TRAIN_MAX_ROWS)]
        rc=_run_streaming(tr, f'TRAIN {slug}/{tag} [{cfg_idx}/{len(SWEEP)}]')
        if rc!=0:
            LEADER.append({'slug':slug,'config':tag,'accepted_prefix':-1.0,
                           'depth1':None,'failed':'train','rc':rc})
            continue
        # Eval.
        eval_depth=min(8,max(4,int(rt.split(',')[-1])))
        ev=['python','-u',EVAL,'--ckpt',os.path.join(ckpt,'latest.npz'),'--frozen',frozen,
            '--corpus',shards,'--out',tj,'--device','cuda',
            '--target-mode','corpus','--chain-hidden','--depth',str(eval_depth),
            '--num-blocks',str(nb),'--head-ff-mult',str(ffm),
            '--max-windows',str(EVAL_MAX_WINDOWS)]
        rc=_run_streaming(ev, f'EVAL  {slug}/{tag}')
        prefix=-1.0; depth1=None
        if rc==0 and os.path.isfile(tj):
            try:
                d=json.load(open(tj))
                prefix=float(d.get('tau',-1.0))
                depth1=d.get('depth1_accept_rate')
            except Exception as e:
                print(f'  !! tau.json unreadable: {e}', flush=True)
        LEADER.append({'slug':slug,'config':tag,'accepted_prefix':prefix,
                       'depth1':depth1})
        print(f'{slug:6s} {tag:32s} accepted_prefix={prefix:.3f} -> ~{1+prefix:.3f}x', flush=True)
        if prefix>best_prefix: best_prefix=prefix; best_dir=ckpt
    if best_dir: BEST[slug]={'dir':best_dir,'accepted_prefix':best_prefix}
    print(f'=== {slug} BEST: {best_dir} accepted_prefix={best_prefix:.3f} ===', flush=True)
print('\nBEST per model:', {k:round(v['accepted_prefix'],3) for k,v in BEST.items()})


## C. SPARSITY PREDICTOR  *(experimental — default OFF; audit first)*
The DENOMINATOR lever: predict which FFN weight blocks are active for a token so the runtime skips the rest (read fewer bytes/token → direct tps gain that multiplies with spec). **Requires a local FFN-activation capture** (`dismantle ... --capture-ffn <path>`, NOT yet built — see the after-steps). Set `RUN_SPARSITY=True` only once that capture exists and is uploaded as `<slug>/ffn_act_shards/`. Trainer:
`colab/sparsity_predictor_train.py` (committed in the repo).

Expected parquet schema (one row per sequence, per layer):
```
layer              : int32
norm_in_q          : bytes int8  (n_tok × hidden)   RMSNorm(residual) input
norm_in_scale      : f32
act_blockmax_q     : bytes int8  (n_tok × n_blocks) per-block max|silu*up|
act_blockmax_scale : f32
shape              : list[int]   [n_tok, hidden, n_blocks]
```


In [ ]:
RUN_SPARSITY=False  # AUDIT: enable only when local FFN-activation capture exists
if RUN_SPARSITY:
    SP='/content/dismantle/colab/sparsity_predictor_train.py'
    import subprocess,os
    for slug in READY:
        ffn=os.path.join(DATA_ROOT,slug,'ffn_act_shards')
        frozen=os.path.join(DATA_ROOT,slug,'frozen_gguf.npz')
        if not os.path.isdir(ffn): print(slug,'no ffn capture; skip'); continue
        out=os.path.join(OUT_ROOT,'sparsity',slug); os.makedirs(out,exist_ok=True)
        r=subprocess.run(['python',SP,'--ffn-dir',ffn,'--frozen',frozen,'--out-dir',out,
                           '--device','cuda','--epochs','6','--block-size','256'],
                          capture_output=True,text=True)
        print(slug, 'sparsity:', r.stdout[-400:] if r.returncode==0 else r.stderr[-400:])
else:
    print('sparsity track OFF (default). See after-steps for the local FFN capture.')


## D. Q2/Q3 DISTILLED DRAFT  *(experimental — default OFF; audit first)*
A cheaper standalone draft (small + low-bit) so the runtime can afford deeper/wider TREE drafts. The chained head already serves as the draft; this is a stretch lever. Default OFF; documented for the audit. A distillation harness would train a tiny model to mimic the target's next-token distribution on the corpus, then quantize to Q3_K/Q2_K.

In [ ]:
RUN_Q2_DRAFT=False  # AUDIT: design + validate before spending compute
print('Q2 draft track OFF (default). Stretch lever; the chained head is the primary draft.')


## E. Emit best heads + leaderboard + manifest

In [ ]:
import json,os,shutil,hashlib
def sha(p):
    h=hashlib.sha256()
    with open(p,'rb') as f:
        for b in iter(lambda:f.read(1<<20),b''): h.update(b)
    return h.hexdigest()
FINAL=os.path.join(OUT_ROOT,'best_heads'); os.makedirs(FINAL,exist_ok=True)
entries=[]
for slug,info in BEST.items():
    src=os.path.join(info['dir'],'head_final.safetensors')
    if not os.path.isfile(src):
        print(f'{slug}: WARN no head_final.safetensors at {src}'); continue
    dstd=os.path.join(FINAL,slug); os.makedirs(dstd,exist_ok=True)
    dst=os.path.join(dstd,'head_final.safetensors')
    tmp=dst+'.tmp'; shutil.copy2(src,tmp); os.replace(tmp,dst)
    entries.append({'slug':slug,'label':MODELS[slug]['label'],
                    'capture_layer':MODELS[slug]['capture_layer'],
                    'accepted_prefix':info['accepted_prefix'],
                    'projected_speedup':round(1+info['accepted_prefix'],3),
                    'head_path':dst,'head_sha256':sha(dst)})
manifest_path=os.path.join(FINAL,'manifest.json')
tmp_m=manifest_path+'.tmp'
with open(tmp_m,'w') as f:
    json.dump({'schema':'dismantle-final-push-v1','entries':entries,'leaderboard':LEADER}, f, indent=2)
os.replace(tmp_m, manifest_path)
print('=== FINAL HEADBANK ===')
for e in entries: print(f"{e['slug']:6s} accepted_prefix={e['accepted_prefix']:.3f} -> ~{e['projected_speedup']}x  {e['head_path']}")
print(f'\nFull leaderboard + heads in {FINAL}')
print('\nLocal runtime usage:')
for e in entries:
    print(f'  DISMANTLE_QWEN_EAGLE5_HEAD={e["head_path"]} dismantle generate ...')
